# Experiment: Hops Visualization


This notebook condenses `hops_visualization.ipynb` into reusable hop-count and path-export steps. It compares the PhN, PSO-SA, and MxLbN collections using shared Python functions.


In [ ]:
from pathlib import Path
import sys

from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError("Could not locate the standalone project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from giakoumas_connectome.data import discover_project_root, load_repository_tables
from giakoumas_connectome.hops import (
    build_target_path_tables,
    export_target_path_tables,
    load_default_hop_collections,
    plot_hop_grid,
)
from giakoumas_connectome.plots import save_figure

import matplotlib.pyplot as plt


## Build The Shared Hop Inputs


In [ ]:
PROJECT_ROOT = discover_project_root(Path.cwd())
OUTPUT_DIR = PROJECT_ROOT / 'output' / 'hops_visualization'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tables = load_repository_tables(PROJECT_ROOT)
collections = load_default_hop_collections(PROJECT_ROOT)

collections.keys()


## Hop Counts To Motor And Endocrine


In [ ]:
fig, _ = plot_hop_grid(
    collections,
    tables.connections,
    tables.classification_other,
)
plt.show()


## Motor Paths Up To 2 Hops


In [ ]:
motor_path_tables = build_target_path_tables(
    collections,
    tables.connections,
    tables.classification_other,
    target_class='motor',
    max_hops=2,
)

sample_key = 'StN-SA:Set 1' if 'StN-SA:Set 1' in motor_path_tables else next(iter(motor_path_tables))
print(sample_key)
display(motor_path_tables[sample_key].head())


## Exported Outputs


In [ ]:
figure_path = save_figure(fig, OUTPUT_DIR / 'figures' / 'hops_to_motor_and_endocrine_grid.svg')
plt.close(fig)

path_exports = export_target_path_tables(
    motor_path_tables,
    OUTPUT_DIR / 'tables' / 'motor_paths_up_to_2hops',
)

sorted(
    [str(figure_path.relative_to(PROJECT_ROOT))]
    + [str(path.relative_to(PROJECT_ROOT)) for path in path_exports.values()]
)
